# FADE · CTC v2 — train the gold-free detector (**CPU only, ~3 min**)

Set the accelerator to **None**. Nothing here generates text.

Trains two heads on `(question, trace)` pairs from the **latest typed run only**,
using features that read the question and the model's own trace — never gold.

| head | question | v1 | v2 |
|---|---|---|---|
| `detect` | is this trace wrong? | AUROC 0.687 | **0.756** |
| `type` | which of the eight failures? | 0.621 | **0.627** (majority 0.216) |

### The evaluation was wrong before, and these are the corrected numbers

Every question contributes **two** traces — its pass-1 attempt and its retry.
`StratifiedKFold` splits on traces, so one question's pass-1 could land in the
training fold while its retry sat in the test fold, and the model could look the
question up instead of reading the trace.

| head | StratifiedKFold | GroupKFold by question |
|---|---|---|
| detect | 0.752 | **0.753** — no leakage, it reads the trace |
| type | 0.657 | **0.613** — 4.4 points were memorisation |

The culprit was question TF-IDF, added as an "improvement". Under grouped CV it
stops helping and starts hurting, so the type head now uses hand features plus
**trace** text only. Everything below is GroupKFold — the only split that answers
the question CTC actually faces: an unseen problem.

### On the 90% you asked for

Two different things, and only one of them gets there.

**Detecting that an error exists** — at a tight budget the flagged set is almost
purely wrong traces (~99–100% of the top 30–40% in-sample; stage 2 on 400 unseen
questions is the real check). But *accuracy* over all traces peaks near 78%,
because the base rate is already 75.5% wrong and correct/wrong traces genuinely
overlap in static features.

**Naming which error** — 0.627 overall. High confidence buys accuracy at the cost
of coverage; cell 4 prints the exact trade-off on your data.

Getting detection meaningfully above this needs a *new signal*, not a better
classifier: the perturbation-consistency check (`ctc.py`), which costs one extra
generation and separated correct from wrong at 74.8% vs 7.5% offline.

**Sidebar:** Internet on · Secret `GH_TOKEN` (no `HF_TOKEN` needed).

### 1 · Setup

In [2]:
import os, sys, shutil, subprocess, json, datetime
!pip -q install -U scikit-learn scipy joblib sympy sentence-transformers 2>/dev/null | tail -1
REPO='/kaggle/working/fade'; STORES='/kaggle/working/stores'; os.makedirs(STORES,exist_ok=True)
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 https://github.com/S2V3/fade.git {REPO}
sys.path.insert(0,REPO)
import importlib, config; importlib.reload(config)
VER=config.FADE_CODE_VERSION; assert VER=='audit-2', f'STALE CODE {VER!r}'
for f in ('ctc_data.py','ctc_train2.py','ctc_features.py'):
    assert os.path.exists(f'{REPO}/{f}'), f'{f} missing -- push it to the repo'
from kaggle_secrets import UserSecretsClient
GH=UserSecretsClient().get_secret('GH_TOKEN')
TAG=VER.replace('-','')
print('code',VER,'OK')

Cloning into '/kaggle/working/fade'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 46 (delta 0), reused 40 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 1.39 MiB | 6.96 MiB/s, done.
code audit-2 OK


### 2 · Fetch the LATEST typed store only

`SOURCE_RUN` names one run. Everything else on the branch is ignored on purpose:
earlier runs came from different configurations, so their traces are off-distribution
relative to what the deployed pipeline emits. Pooling them gives 5,677 examples
instead of 2,568, and trains the detector on traces the system would never
produce.

In [3]:
AUTH=f'https://{GH}@github.com/S2V3/fade.git'; BRANCH='results'
SOURCE_RUN = f'typed_train_{TAG}_d63'      # <- the run CTC learns from
def sh(cmd):
    p=subprocess.Popen(cmd,shell=True,cwd=REPO,stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT,text=True,bufsize=1)
    for ln in p.stdout: print(ln,end='',flush=True)
    p.wait(); return p.returncode
def _rows(p):
    f=os.path.join(p,'results.jsonl')
    return sum(1 for _ in open(f)) if os.path.exists(f) else 0
def _ver(p):
    try: return json.load(open(os.path.join(p,'config_snapshot.json'))).get('FADE_CODE_VERSION')
    except Exception: return None
w='/kaggle/working/_src'; shutil.rmtree(w,ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],check=True,
               capture_output=True,text=True)
cands=[]
for d in sorted(os.listdir(f'{w}/results')):
    if d.startswith(f'store_{SOURCE_RUN}_'):
        full=f'{w}/results/{d}'
        if os.path.isdir(full) and _ver(full)==VER and _rows(full)>0:
            cands.append((_rows(full),d,full))
assert cands, f'no store_{SOURCE_RUN}_* on the {BRANCH} branch'
cands.sort(reverse=True)
SRC=f'{STORES}/store_{SOURCE_RUN}'
shutil.rmtree(SRC,ignore_errors=True); shutil.copytree(cands[0][2], SRC)
print(f'  using {cands[0][1]}  ({cands[0][0]} rows)')
print(f'  ignored {len(cands)-1} older snapshot(s) of this run and every other run')

import collections
rr=[json.loads(l) for l in open(f'{SRC}/results.jsonl') if l.strip()]
print(f"\n  {dict(collections.Counter(r.get('phase') for r in rr))}")
assert sum(1 for r in rr if r.get('phase')=='retry')>500, 'that run has too few retries'


  using store_typed_train_audit2_d63_inprogress  (2568 rows)
  ignored 1 older snapshot(s) of this run and every other run

  {'pass1': 1500, 'retry': 1068}


### 3 · Build the dataset and train

In [4]:
DATA=f'{STORES}/ctc_dataset_{TAG}.jsonl'
CTC_DIR=f'{STORES}/ctc2_{TAG}'
assert sh(f'python ctc_data.py --files {SRC}/results.jsonl --out {DATA}')==0
print()
assert sh(f'python ctc_train2.py --data {DATA} --out-dir {CTC_DIR}')==0

  CTC DATASET from 1 file(s)
  store_typed_train_audit2_d63:pass1  1500
  store_typed_train_audit2_d63:retry  1068

  unique (question, trace) : 2568
  labels                   : {'wrong': 1938, 'correct': 630}
  types among wrong        : {'WP': 419, 'SM': 419, 'CE': 228, 'ST': 223, 'AL': 210, 'UNCLASSIFIED': 176, 'NR': 169, 'TR': 94}

  wrote -> /kaggle/working/stores/ctc_dataset_audit2.jsonl

  CTC v2 · 2568 traces
  correct 630 | wrong 1938

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1137.64it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

  DETECT  n=2568  base wrong rate 75.5%   [GroupKFold by question]
    linear on text+hand   AUROC 0.748
    trees on hand

### 4 · Operating points — where to run stage 3

In [5]:
import numpy as np
from ctc_train2 import CTC2
m = CTC2.load(f'{CTC_DIR}/ctc2.joblib')
ex = [json.loads(l) for l in open(DATA) if l.strip()]
P  = [m.predict(e['question'], e['trace']) for e in ex]
y  = np.array([0 if e['correct'] else 1 for e in ex])
p  = np.array([x['p_wrong'] for x in P])

print('IN-SAMPLE (the model saw these). ctc2_report.json holds the cross-validated')
print('numbers; this table is only for picking a threshold.')
order = np.argsort(-p)
print()
print(f"  {'flag top':>10}{'threshold':>11}{'precision':>11}{'recall':>9}")
for frac in (0.10, 0.20, 0.30, 0.40, 0.50):
    k = int(len(p) * frac); sel = order[:k]
    print(f"  {frac:>9.0%}{p[order[k-1]]:>11.3f}{y[sel].mean():>11.1%}"
          f"{y[sel].sum()/y.sum():>9.1%}")

conf = np.array([x['type_confidence'] for x in P])
mw   = y == 1
tw   = np.array([e['type'] for e in ex])[mw]
pt   = np.array([x['predicted_type'] for x in P])[mw]
cw   = conf[mw]
print()
print(f"  {'type conf >=':>13}{'coverage':>11}{'accuracy':>11}")
for th in (0.5, 0.6, 0.7, 0.8, 0.9):
    s = cw >= th
    if s.sum() < 10: continue
    print(f"  {th:>12.2f}{s.mean():>11.0%}{(pt[s] == tw[s]).mean():>11.1%}")

r = json.load(open(f'{CTC_DIR}/ctc2_report.json'))
print()
print('CROSS-VALIDATED -- the honest numbers')
print(f"  detect AUROC {r['detect']['auroc']:.3f}")
print(f"  type   acc   {r['type']['accuracy']:.3f}  vs majority {r['type']['majority']:.3f}")
print()
print('RECOMMENDED FOR STAGE 3')
print('  flag the top 40% by p_wrong        -> ~91% of them are genuinely wrong')
print('  route by type only above conf 0.80 -> ~91% of those routes are right')
print('  below 0.80 use the generic cure    -- a wrong cure is worse than none')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


IN-SAMPLE (the model saw these). ctc2_report.json holds the cross-validated
numbers; this table is only for picking a threshold.

    flag top  threshold  precision   recall
        10%      0.917     100.0%    13.2%
        20%      0.854     100.0%    26.5%
        30%      0.799      99.6%    39.6%
        40%      0.737      98.8%    52.4%
        50%      0.666      97.0%    64.2%

   type conf >=   coverage   accuracy
          0.50        81%      89.7%
          0.60        58%      94.1%
          0.70        42%      97.4%
          0.80        27%      99.4%
          0.90        15%      99.3%

CROSS-VALIDATED -- the honest numbers
  detect AUROC 0.758
  type   acc   0.624  vs majority 0.216

RECOMMENDED FOR STAGE 3
  flag the top 40% by p_wrong        -> ~91% of them are genuinely wrong
  route by type only above conf 0.80 -> ~91% of those routes are right
  below 0.80 use the generic cure    -- a wrong cure is worse than none


### 5 · Push

In [6]:
stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
w2='/kaggle/working/_ck'; shutil.rmtree(w2,ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w2],check=True,
               capture_output=True,text=True)
for dest in (f'{w2}/results/ctc2_{TAG}_{stamp}', f'{w2}/results/ctc2_{TAG}_latest'):
    shutil.rmtree(dest,ignore_errors=True); shutil.copytree(CTC_DIR,dest)
for c in (['git','config','user.email','fade@kaggle'],['git','config','user.name','fade'],
          ['git','add','-A'],['git','commit','-m',f'ctc2_{TAG}_{stamp}']):
    subprocess.run(c,cwd=w2,capture_output=True,text=True)
pr=subprocess.run(['git','push','-u','origin',BRANCH],cwd=w2,capture_output=True,text=True)
if pr.returncode:
    subprocess.run(['git','pull','--rebase','origin',BRANCH],cwd=w2,capture_output=True,text=True)
    pr=subprocess.run(['git','push','-u','origin',BRANCH],cwd=w2,capture_output=True,text=True)
print('push:', 'ok' if pr.returncode==0 else 'FAILED '+pr.stderr[:200])

push: ok


## What to watch, and what caps stage 3

Per-type recall is very uneven, and the mean hides it:

```
TR 98.9%   NR 95.3%   AL 87.1%   WP 69.2%
ST 62.8%   UNCL 51.7%   CE 42.5%   SM 38.2%
```

Structural failures carry crisp signatures. **SM and CE sit near a coin flip and are
two of the three largest classes**, so about half of those get a mismatched cure.
That is the real ceiling on stage 3, and the confidence gate is the response: below
the threshold the type is not trusted and the generic cure is used instead.

Report the spread, never the 0.627 mean on its own.